# FlashAttention lab: existing Triton tests → benchmarks

Upload this notebook to [Google Colab](https://colab.research.google.com/), choose **Runtime → Change runtime type → GPU**, then run cells in order. Nothing has been executed remotely in the saved notebook. No Modal dependency or Modal job is used.

The notebook embeds the current source and **unchanged tests**, with SHA-256 checksums; no GitHub URL, upload, Drive mount, or credentials are required. Rebuild after source changes using `python scripts/build_colab_notebook.py`.

**Compatibility is checked before installing dependencies.** The existing kernel requires NVIDIA **SM80+**. A T4 (SM75), P100 (SM60), V100 (SM70), CPU, or TPU runtime stops with an explanation. An A100 or L4 meets the architecture requirement, but Colab does not guarantee a particular GPU or free access. This pinned stack also conservatively requires Linux x86_64, Python 3.10–3.13, and NVIDIA driver **570.26+** for CUDA 12.8. No driver is installed or changed.

Dependencies go into a private virtual environment: torch **2.8.0+cu128**, Triton **3.4.0**, pytest **8.3.4**, matplotlib **3.10.3**. The notebook kernel's packages are left alone. Tests and benchmarks use the isolated interpreter and `PYTHONPATH=src`; the project's original Python 3.13 packaging requirement is preserved. The actual Colab Python version is recorded, so comparisons with the Mac environment must account for that difference.

The numerical tests run first. A successful pytest exit **and all 16 distinct GPU correctness cases passing with zero skips** are required before benchmarks. Any installation failure, incompatible assignment, test failure, missing report, or timeout stops progress. **On failure, run the last download cell manually** to save the explanation and available logs.

References: [Colab resource availability](https://research.google.com/colaboratory/faq.html), [NVIDIA compute capabilities](https://developer.nvidia.com/cuda/gpus), [CUDA 12.8 driver requirements](https://docs.nvidia.com/cuda/archive/12.8.0/cuda-toolkit-release-notes/), [PyTorch 2.8 CUDA wheels](https://pytorch.org/get-started/previous-versions/).


## 1. Detect the GPU and reject incompatible assignments

This cell uses only the standard library and `nvidia-smi`. It creates a fresh workspace and saves preflight results even when compatibility fails.

In [ ]:
# Standard library only: GPU detection happens before pip or torch imports.
import base64, hashlib, io, zipfile
PAYLOAD = base64.b64decode(
    '...'
)
SOURCE_SHA256 = '8c2e4769e13cfc28d582aa37b02a65fe19f9e82c43fd4f086b7ed6c2bc19da02'
assert hashlib.sha256(PAYLOAD).hexdigest() == SOURCE_SHA256, 'Embedded source checksum mismatch'
with zipfile.ZipFile(io.BytesIO(PAYLOAD)) as archive:
    support_source = archive.read('scripts/colab_support.py')
support = {}
exec(compile(support_source, 'colab_support.py', 'exec'), support)
context = support['initialize'](PAYLOAD, SOURCE_SHA256)
support['preflight'](context)


## 2. Install the pinned stack in an isolated environment

The GPU is checked again before setup. Install logs are saved. Each setup command has a ten-minute timeout; setup can consume Colab runtime time and network bandwidth. No package installation occurs if preflight fails.

In [ ]:
support['install_dependencies'](context)


## 3. Run the existing tests, then the benchmark sweep

Runs the exact embedded `tests/test_triton.py`, comparing fp16/bf16 causal and noncausal outputs against CPU float64 and CUDA math SDPA. Covers partial tiles, odd dimensions, multiple batches/heads, and future isolation. The existing shared correctness gate blocks the benchmark callback unless every GPU case passes.

The `nvidia` benchmark profile sweeps sequence length through 2048, head dimension, independent query/KV tiles, batch size, and head count, plus causal/partial cases. Compares standard, **educational PyTorch blockwise**, default-dispatch SDPA, and **fused Triton**. Three warmups, ten synchronized timings, first-call cost separately, numerical checks for every case. Actual backend evidence is recorded or labeled unknown. The Triton test stage warms kernel caches; benchmark first-call timing does not promise cold compilation.

The worker has a **600-second total timeout**, including a 240-second correctness subprocess limit. On timeout/interruption the worker process group is stopped and available files are bundled. This does not disconnect the Colab runtime or impose a monetary cap. Disconnect/delete the runtime when finished. Results report CUDA allocator peaks, not total GPU/system memory. Every retry gets a separate artifact directory; earlier measurements cannot masquerade as the latest attempt.


In [ ]:
support['run_experiment'](context)


## 4. Download all results — also usable after a failure

The ZIP includes the embedded source snapshot and hashes, hardware/driver report, status, installation log, package versions, correctness logs/JUnit, worker log, and any completed/partial benchmark JSON/CSV plus latency/memory PNG/SVG plots. Failed or incompatible runs contain explanations rather than invented measurements. If the browser blocks automatic download, use Colab's Files panel to download the printed ZIP path. Download before the runtime disconnects; local Colab storage is temporary.


In [ ]:
if 'context' not in globals():
    raise RuntimeError('Run the first cell to create a source snapshot and preflight report.')
archive_path = support['bundle'](context)
from google.colab import files
files.download(archive_path)
